In [1]:
from qldpc_sim import *
from qldpc_sim.qldpc_experiment import *
from qldpc_sim.qec_code import *
from qldpc_sim.data_structure import *
from qldpc_sim.ckbb_surgery import *
import stim
import numpy as np

In [2]:
def setup_cnot_exp():
    patch1 = RotatedSurfaceCode.from_distance(3, code_name="ancilla", system_coordinate=(0,0))
    patch2 = RotatedSurfaceCode.from_distance(3, code_name="control", system_coordinate=(0,1))
    patch3 = RotatedSurfaceCode.from_distance(3, code_name="target", system_coordinate=(0,2))
    qm = qldpc_experiment.QuantumMemory(size=600)
    patches = [patch1, patch2, patch3]
    mapqb = {}
    for c in patches:
        for lq in c.logical_qubits:
            mapqb[lq.logical_x] = c
            mapqb[lq.logical_z] = c

    ctx = Context(
        codes=[patch1, patch2, patch3], logical_qubits=patch1.logical_qubits + patch2.logical_qubits + patch3.logical_qubits, initial_assignement=mapqb, memory=qm
    )

    return ctx

In [3]:
context = setup_cnot_exp()

ancilla = context.codes[0]
control = context.codes[1]
target = context.codes[2]

cnot_program = (
    [
        InitializeCode(
            code=control,
            context=context,
            tag=f"init_{control.id}",
            initial_state=PauliEigenState.Z_plus,
        )
    ]
    + [
        InitializeCode(
            code=target,
            context=context,
            tag=f"init_{target.id}",
            initial_state=PauliEigenState.Z_plus,
        )
    ]
    + [
        InitializeCode(
            code=ancilla,
            context=context,
            tag=f"init_{ancilla.id}",
            initial_state=PauliEigenState.X_plus,
        )
    ]
    + [
        StabMeasurement(code=p, context=context, tag=f"isb_2_{p.id}", round=3)
        for p in [control, ancilla, target]
    ]
    + [
        CKBBMeasurement(
            distance=3,
            context=context,
            tag="Merge_C-A",
            logical_targets=[
                control.logical_qubits[0].logical_z,
                ancilla.logical_qubits[0].logical_z,
            ],
        ),
    ]
    + [
        StabMeasurement(code=p, context=context, tag=f"isb_2_{p.id}", round=3)
        for p in [control, ancilla, target]
    ]
    + [
        CKBBMeasurement(
            distance=3,
            context=context,
            tag="Merge_T-A",
            logical_targets=[
                ancilla.logical_qubits[0].logical_x,
                target.logical_qubits[0].logical_x,
            ],
        ),
    ]
    + [
        StabMeasurement(code=p, context=context, tag=f"isb_3_{p.id}", round=3)
        for p in [control, ancilla, target]
    ]
    + [
        LM(
            logical_targets=[ancilla.logical_qubits[0].logical_z],
            context=context,
            tag=f"final_ancilla",
            basis=PauliChar.Z,
        ),
        LM(
            logical_targets=[target.logical_qubits[0].logical_z],
            context=context,
            tag=f"final_target",
            basis=PauliChar.Z,
        ),
        LM(
            logical_targets=[control.logical_qubits[0].logical_z],
            context=context,
            tag=f"final_control",
            basis=PauliChar.Z,
        ),
    ]
)

In [4]:
from typing import Dict, List, Set, Tuple

from qldpc_sim.qldpc_experiment.qec_gadget import LogicGadget


def _condition_item_label(item) -> str:
    tag = getattr(item, "tag", None)
    item_id = getattr(item, "id", None)

    if tag is not None and item_id is not None:
        return f"{tag}({str(item_id)[:4]})"
    if tag is not None:
        return str(tag)
    return str(item)


def _logical_qubit_label(logical_qubit: LogicalQubit) -> str:
    name = getattr(logical_qubit, "name", None)
    if name:
        return str(name)

    qubit_id = getattr(logical_qubit, "id", None)
    if qubit_id is not None:
        return str(qubit_id)

    return str(logical_qubit)


def print_frame_tracker_state(
    frame_tracker: FrameState,
    header: str = "Frame tracker state",
) -> None:
    print(f"\n{header}:")

    if not frame_tracker.frame_corrections:
        print("  <empty>")
        return

    for logical_qubit, correction in sorted(
        frame_tracker.frame_corrections.items(),
        key=lambda item: _logical_qubit_label(item[0]),
    ):
        x_items = sorted(
            _condition_item_label(node) for node in correction.correction_X_cond
        )
        z_items = sorted(
            _condition_item_label(node) for node in correction.correction_Z_cond
        )

        print(f"  {_logical_qubit_label(logical_qubit)}")
        print(f"    X ({len(x_items)}): {x_items}")
        print(f"    Z ({len(z_items)}): {z_items}")


def evaluate(
    ctx,
    program: List[QECGadget],
    verbose: bool = True,
) -> Tuple[List[str], Dict[str, Set[int]], MeasurementRecord]:
    """Evaluate a qLDPC program in the context."""
    logical_outcomes: Dict[str, Set[int]] = {}
    instructions = []

    for gadget_index, gadget in enumerate(program, start=1):
        compilers, outcomes = gadget.build_compiler_instructions()

        for compiler_index, compiler in enumerate(compilers, start=1):
            n_instructions, measurements = compiler.compile(ctx.memory)
            instructions.extend(n_instructions)

            if measurements is not None:
                ctx.record.add_measurements(measurements)

            if verbose:
                n_meas = 0 if measurements is None else len(measurements.measured_nodes)

        for outcome_index, outcome in enumerate(outcomes, start=1):
            ctx.record.add_event(outcome)

            if outcome.type == EventType.FRAME_CORRECTION:
                logical_qubit = ctx.map_operator_to_qubits[outcome.target]
                ctx.frame_tracker.add_correction(
                    logical_qubit, outcome.target.logical_type, outcome.measured_nodes
                )

            if outcome.type == EventType.OBSERVABLE:
                # Keep per-tag node support in XOR form: if a node appears twice, it cancels.
                if outcome.tag not in logical_outcomes:
                    logical_outcomes[outcome.tag] = set()

                direct_nodes = set(outcome.measured_nodes)

                target_ops = []
                if isinstance(outcome.target, LogicalOperator):
                    target_ops = [outcome.target]
                elif isinstance(outcome.target, set):
                    target_ops = [
                        op for op in outcome.target if isinstance(op, LogicalOperator)
                    ]

                # For multi-logical observables (e.g., ZZ or XX), combine correction
                # contributions from all involved logicals with XOR.
                merged_nodes = set()
                for target_op in target_ops:
                    logical_qubit = ctx.map_operator_to_qubits[target_op]
                    tracked_correction = ctx.frame_tracker.get_correction(
                        logical_qubit, target_op.logical_type
                    )

                    merged_nodes ^= tracked_correction
                observable_nodes = direct_nodes ^ merged_nodes
                logical_outcomes[outcome.tag] = observable_nodes

    print_frame_tracker_state(ctx.frame_tracker, header="Final frame tracker state")

    return instructions, logical_outcomes, ctx.record

In [5]:
from collections import Counter
from qldpc_sim.qldpc_experiment.interpreter import concat_events_per_sample, xor_event_nodes

ctx = context
ctx.record = MeasurementRecord()
ctx.frame_tracker = FrameState()
ctx.memory = QuantumMemory(size=600)
prog = cnot_program

print("Reset context state before debug run:")
print(f"  record measurements: {len(ctx.record.measurements)}")
print(f"  record events: {len(ctx.record.events)}")
print(f"  frame entries: {len(ctx.frame_tracker.frame_corrections)}")

p, lo, rec = evaluate(ctx, prog, verbose=True)

# Run stim and compute logical outcomes directly from lo and rec.
num_shots = 30
shots = stim.Circuit("\n".join(p)).compile_sampler().sample(shots=num_shots).astype(np.uint8)

# rec_idx(node): latest record index for each measured node.
rec_idx = {node: idxs[-1] for node, idxs in rec.measurements.items() if idxs}

# For each logical support l in lo: outcome = XOR over shots[:, rec_idx(i)] for i in l.
logical_record_idx = {}
logical_outcomes = {}
for tag, l in lo.items():
    idx = sorted({rec_idx[i] for i in l if i in rec_idx})
    logical_record_idx[tag] = idx
    if idx:
        logical_outcomes[tag] = np.bitwise_xor.reduce(shots[:, idx], axis=1).astype(np.uint8)
    else:
        logical_outcomes[tag] = np.zeros(num_shots, dtype=np.uint8)

print(f"Stim shots: {num_shots}")
print("logical_record_idx:")
for tag in sorted(logical_record_idx):
    print(f"  {tag}: {logical_record_idx[tag]}")

print("\nlogical_outcomes (first 10 shots):")
for tag in sorted(logical_outcomes):
    print(f"  {tag}: {logical_outcomes[tag][:10].tolist()}")

Reset context state before debug run:
  record measurements: 0
  record events: 0
  frame entries: 0

Final frame tracker state:
  ancilla_lq_0
    X (1): ['_c_x_0_ancilla_T(f5d2)']
    Z (2): ['_c_z_2_target_T(4891)', 'bv_l0(c206)']
  control_lq_0
    X (1): ['bv_l0(190a)']
    Z (0): []
  target_lq_0
    X (0): []
    Z (0): []
Stim shots: 30
logical_record_idx:
  LogicalMeasurement_final_ancilla: [464, 481, 558, 559, 560]
  LogicalMeasurement_final_control: [564, 565, 566]
  LogicalMeasurement_final_target: [561, 562, 563]
  Merge_C-A_parity_outcome: [184, 187, 193, 195, 196, 198, 199, 202, 206, 209, 211, 212, 214, 217, 218, 222, 223, 227]
  Merge_T-A_parity_outcome: [240, 418, 423, 426, 427, 436, 437, 438, 439, 441, 444, 445, 446, 448, 449, 452, 456, 457, 458]

logical_outcomes (first 10 shots):
  LogicalMeasurement_final_ancilla: [1, 1, 1, 1, 0, 1, 0, 0, 0, 1]
  LogicalMeasurement_final_control: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  LogicalMeasurement_final_target: [0, 1, 0, 1, 0, 0, 1

In [6]:
def resolve_tag(name: str) -> str:
    if name in logical_outcomes:
        return name

    matches = [
        key
        for key in logical_outcomes
        if key == name
        or key.endswith(f"_{name}")
        or key.startswith(f"{name}_")
        or f"_{name}_" in key
    ]

    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise KeyError(f"Could not find tag matching '{name}'. Available tags: {sorted(logical_outcomes)}")
    raise KeyError(f"Ambiguous tag '{name}'. Matches: {matches}")

final_target_tag = resolve_tag("final_target")
final_ancilla_tag = resolve_tag("final_ancilla")
merge_ca_tag = resolve_tag("Merge_C-A")
final_control_tag = resolve_tag("final_control")

final_target_corr = (
    logical_outcomes[final_target_tag]
    ^ logical_outcomes[final_ancilla_tag]
    ^ logical_outcomes[merge_ca_tag]
).astype(np.uint8)

joint_corr = Counter(
    zip(
        logical_outcomes[final_control_tag].tolist(),
        final_target_corr.tolist(),
    )
)

print("Resolved tags:")
print(f"  final_target -> {final_target_tag}")
print(f"  final_ancilla -> {final_ancilla_tag}")
print(f"  Merge_C-A -> {merge_ca_tag}")
print(f"  final_control -> {final_control_tag}")

print("\n(final_control, final_target_corr): count")
for outcome, count in sorted(joint_corr.items()):
    print(f"  {outcome}: {count}")

Resolved tags:
  final_target -> LogicalMeasurement_final_target
  final_ancilla -> LogicalMeasurement_final_ancilla
  Merge_C-A -> Merge_C-A_parity_outcome
  final_control -> LogicalMeasurement_final_control

(final_control, final_target_corr): count
  (0, 0): 30


In [7]:
from collections import Counter

tags = sorted(logical_outcomes)
joint = Counter(zip(*(logical_outcomes[t].tolist() for t in tags)))
print("(" + ", ".join(tags) + "): count")
for outcome, count in sorted(joint.items()):
    print(f"  {outcome}: {count}")


(LogicalMeasurement_final_ancilla, LogicalMeasurement_final_control, LogicalMeasurement_final_target, Merge_C-A_parity_outcome, Merge_T-A_parity_outcome): count
  (0, 0, 0, 0, 0): 4
  (0, 0, 0, 0, 1): 3
  (0, 0, 1, 1, 0): 4
  (0, 0, 1, 1, 1): 1
  (1, 0, 0, 1, 0): 4
  (1, 0, 0, 1, 1): 8
  (1, 0, 1, 0, 0): 2
  (1, 0, 1, 0, 1): 4
